# Gemma 4 31B OCR-VQA 3단계 QLoRA 학습 — Colab G4 96GB

이 노트북은 기존 3단계 OCR curriculum을 **`google/gemma-4-31B-it`**와 Colab G4(RTX PRO 6000 96GB)에 맞춰 구성한 별도 버전입니다.

1. Stage 1 — 기본 데이터: `ssafy-16-2-ai/train.csv`
2. Stage 2 — 증강 데이터: `final_data/modify_50000/train_all_augmented.csv`
3. Stage 3 — 추가 데이터: 영어, 색상·개수, 중국어, 일본어, 비영어 데이터

각 Stage는 독립 checkpoint 폴더에 실제 LoRA 파라미터 값, 파라미터 manifest, 학습 설정, 데이터 fingerprint, 검증 결과를 저장합니다. 학습 완료 후 최종 Stage adapter로 `test.csv`를 추론하여 제출 CSV와 상세 CSV를 저장합니다.

- 모델: `google/gemma-4-31B-it` (31B dense multimodal)
- 학습: NF4 4-bit QLoRA, BF16 compute, rank-16 LoRA
- 대상 GPU: Colab G4 / NVIDIA RTX PRO 6000 96GB
- 모델 카드: https://huggingface.co/google/gemma-4-31B-it

> L4/T4용 노트북이 아닙니다. GPU 메모리가 75GiB 미만이면 코드가 실행을 중단합니다.


Each completed Stage also runs test.csv with its own adapter and saves a separate Kaggle submission CSV.

## 실행 순서

1. Colab에서 GPU 런타임을 선택합니다.
2. 패키지 설치 셀 실행 후 런타임을 한 번 재시작합니다.
3. `ssafy-16-2-ai.zip`과 `final_data.zip`을 **압축을 풀지 않은 상태로** Colab `/content`에 업로드합니다.
   - 파일이 크면 Google Drive에 두어도 됩니다. 코드는 `/content`를 먼저 찾고 Drive를 다음으로 찾습니다.
   - Colab 파일 패널 업로드는 런타임 종료 시 사라지므로 학습 checkpoint는 Drive 저장을 권장합니다.
4. ZIP 파일명·비밀번호 설정을 확인하고 위에서 아래로 실행합니다.
5. `SMOKE_TEST=False` 상태에서 각 데이터 단계를 1 epoch씩 전체 학습합니다.

두 ZIP은 각각 별도 디렉터리에 자동 해제됩니다. 압축 내부가 `ssafy-16-2-ai/...` 또는 바로 `train.csv`,
`final_data/...` 또는 바로 `modify_50000/...` 형태여도 실제 데이터 루트를 자동 탐색합니다.

`final_data`의 CSV와 Excel 메타데이터는 공통 8열 형식(`id,path,question,a,b,c,d,answer`)으로 변환됩니다.


## 1. 패키지 설치

Colab 기본 PyTorch는 유지합니다. Gemma 4 31B와 G4 Blackwell QLoRA에 필요한 최신 패키지를 설치합니다.


In [ ]:
!pip -q install -U "transformers>=5.10.1" accelerate peft bitsandbytes datasets scikit-learn openpyxl huggingface_hub


In [2]:
# 설치 직후에는 Runtime > Restart session을 한 번 실행하는 것을 권장합니다.
import importlib.metadata as md
for package in ["transformers", "peft", "accelerate", "bitsandbytes", "openpyxl"]:
    print(f"{package:16s}", md.version(package))


transformers     5.17.0
peft             0.21.0
accelerate       1.15.0
bitsandbytes     0.50.2
openpyxl         3.1.5


## 2. Hugging Face 인증


In [ ]:
# Gemma 모델 페이지에서 라이선스/접근 조건을 먼저 승인한 뒤 로그인하세요.
from huggingface_hub import notebook_login
notebook_login()


## 3. ZIP 준비, 선택적 Drive 마운트, 자동 압축 해제

기본 입력은 Colab에 업로드한 다음 두 파일입니다.

- `/content/ssafy-16-2-ai.zip`
- `/content/final_data.zip`

`MOUNT_GOOGLE_DRIVE=True`이면 checkpoint를 Drive에 영구 저장하며, `/content`에 ZIP이 없을 때
`/content/drive/MyDrive`에서도 같은 파일명을 찾습니다.


In [4]:
from pathlib import Path

# checkpoint를 런타임 종료 후에도 보존하려면 True를 권장합니다.
# ZIP을 /content에 직접 올렸더라도 Drive는 결과 저장용으로만 사용할 수 있습니다.
MOUNT_GOOGLE_DRIVE = True

if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
else:
    DRIVE_ROOT = None

print("Drive mounted:", DRIVE_ROOT is not None)


Mounted at /content/drive
Drive mounted: True


In [5]:
import json
import os
import shutil
import subprocess
from pathlib import Path

# ------------------------------------------------------------
# ZIP 설정
# ------------------------------------------------------------
# None이면 /content를 우선 확인한 뒤 Drive에서 동일 파일명을 자동 탐색합니다.
BASE_ZIP_PATH = None   # 예: Path("/content/ssafy-16-2-ai.zip")
FINAL_ZIP_PATH = None  # 예: Path("/content/final_data.zip")

BASE_ZIP_NAME = "ssafy-16-2-ai.zip"
FINAL_ZIP_NAME = "final_data.zip"

BASE_ZIP_PASSWORD = os.environ.get("VQA_ZIP_PASSWORD", "")  # 대회 제공 비밀번호는 환경 변수로 전달
FINAL_ZIP_PASSWORD = None  # final_data.zip에도 암호가 있으면 문자열로 입력

# False가 안전한 기본값입니다. Colab 디스크가 부족하면 True로 바꿔 해제 완료 후 업로드 ZIP을 삭제할 수 있습니다.
DELETE_ZIP_AFTER_EXTRACT = False

WORK_ROOT = Path("/content/vqa_data")
BASE_EXTRACT_ROOT = WORK_ROOT / "base_archive"
FINAL_EXTRACT_ROOT = WORK_ROOT / "final_archive"
for directory in [BASE_EXTRACT_ROOT, FINAL_EXTRACT_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

def resolve_zip(explicit_path, filename: str) -> Path:
    if explicit_path is not None:
        path = Path(explicit_path).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f"지정한 ZIP이 없습니다: {path}")
        return path.resolve()

    candidates = [Path("/content") / filename]
    if DRIVE_ROOT is not None:
        candidates.append(DRIVE_ROOT / filename)
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()

    # Drive 하위 폴더에 넣은 경우 마지막으로 파일명 검색
    if DRIVE_ROOT is not None and DRIVE_ROOT.exists():
        matches = list(DRIVE_ROOT.rglob(filename))
        if matches:
            return matches[0].resolve()

    checked = "\n".join(f"  - {path}" for path in candidates)
    raise FileNotFoundError(
        f"{filename}을 찾지 못했습니다. Colab /content에 업로드하거나 경로를 직접 지정하세요.\n{checked}"
    )

BASE_ZIP_PATH = resolve_zip(BASE_ZIP_PATH, BASE_ZIP_NAME)
FINAL_ZIP_PATH = resolve_zip(FINAL_ZIP_PATH, FINAL_ZIP_NAME)
print("BASE ZIP :", BASE_ZIP_PATH, f"({BASE_ZIP_PATH.stat().st_size / 1024**3:.2f} GB)")
print("FINAL ZIP:", FINAL_ZIP_PATH, f"({FINAL_ZIP_PATH.stat().st_size / 1024**3:.2f} GB)")

free_gb = shutil.disk_usage("/content").free / 1024**3
print(f"/content free space before extraction: {free_gb:.1f} GB")
if free_gb < 40:
    print("[WARN] 디스크 여유가 적습니다. 압축 해제 크기를 확인하고 필요하면 High-RAM 런타임을 사용하세요.")

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "p7zip-full"], check=True)

def archive_signature(archive: Path):
    stat = archive.stat()
    return {"path": str(archive), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}

def extract_archive(archive: Path, target_root: Path, password=None):
    marker = target_root / ".extraction_complete.json"
    signature = archive_signature(archive)
    if marker.exists():
        try:
            previous = json.loads(marker.read_text(encoding="utf-8"))
        except Exception:
            previous = None
        if previous == signature:
            print("이미 같은 ZIP을 해제했습니다:", archive.name)
            return

    command = ["7z", "x", "-y", f"-o{target_root}"]
    if password:
        command.append(f"-p{password}")
    command.append(str(archive))
    print("Extracting:", archive, "->", target_root)
    subprocess.run(command, check=True)
    marker.write_text(json.dumps(signature, ensure_ascii=False, indent=2), encoding="utf-8")

    if DELETE_ZIP_AFTER_EXTRACT and archive.is_file():
        archive.unlink()
        print("압축 해제 후 ZIP 삭제:", archive)

extract_archive(BASE_ZIP_PATH, BASE_EXTRACT_ROOT, BASE_ZIP_PASSWORD)
extract_archive(FINAL_ZIP_PATH, FINAL_EXTRACT_ROOT, FINAL_ZIP_PASSWORD)

# 이미 다른 위치에 해제해 둔 경우에만 아래 override를 지정합니다.
BASE_ROOT_OVERRIDE = None
FINAL_ROOT_OVERRIDE = None

def discover_base_root() -> Path:
    if BASE_ROOT_OVERRIDE:
        return Path(BASE_ROOT_OVERRIDE).resolve()
    candidates = []
    for csv_path in BASE_EXTRACT_ROOT.rglob("train.csv"):
        parent = csv_path.parent
        if (parent / "train").is_dir() and (parent / "dev.csv").is_file():
            candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(
            f"{BASE_EXTRACT_ROOT} 아래에서 train.csv, dev.csv, train/ 조합을 찾지 못했습니다."
        )
    return sorted(candidates, key=lambda path: (path.name != "ssafy-16-2-ai", len(path.parts)))[0].resolve()

def discover_final_root() -> Path:
    if FINAL_ROOT_OVERRIDE:
        return Path(FINAL_ROOT_OVERRIDE).resolve()

    # 압축 내부에 final_data 상위 폴더가 있는 경우
    named_candidates = [
        path for path in FINAL_EXTRACT_ROOT.rglob("final_data")
        if path.is_dir() and (path / "modify_50000").is_dir()
    ]
    if named_candidates:
        return sorted(named_candidates, key=lambda path: len(path.parts))[0].resolve()

    # ZIP 최상위가 곧 final_data 내용인 경우
    direct_candidates = [FINAL_EXTRACT_ROOT, *[p for p in FINAL_EXTRACT_ROOT.iterdir() if p.is_dir()]]
    for candidate in direct_candidates:
        if (candidate / "modify_50000").is_dir():
            return candidate.resolve()

    raise FileNotFoundError(
        f"{FINAL_EXTRACT_ROOT} 아래에서 modify_50000 폴더를 포함한 final_data 루트를 찾지 못했습니다."
    )

BASE_ROOT = discover_base_root()
FINAL_ROOT = discover_final_root()

required_paths = [
    BASE_ROOT / "train.csv",
    BASE_ROOT / "train",
    FINAL_ROOT / "modify_50000" / "train_all_augmented.csv",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(f"압축 해제 후 필수 데이터 누락: {missing}")

print("BASE_ROOT :", BASE_ROOT)
print("FINAL_ROOT:", FINAL_ROOT)
print(f"/content free space after extraction: {shutil.disk_usage('/content').free / 1024**3:.1f} GB")


BASE ZIP : /content/drive/MyDrive/ssafy-16-2-ai.zip (1.68 GB)
FINAL ZIP: /content/drive/MyDrive/final_data.zip (16.09 GB)
/content free space before extraction: 187.9 GB
Extracting: /content/drive/MyDrive/ssafy-16-2-ai.zip -> /content/vqa_data/base_archive
Extracting: /content/drive/MyDrive/final_data.zip -> /content/vqa_data/final_archive
BASE_ROOT : /content/vqa_data/base_archive
FINAL_ROOT: /content/vqa_data/final_archive
/content free space after extraction: 152.0 GB


## 4. 학습 설정


In [6]:
import gc, hashlib, json, math, random, re, shutil, time
from dataclasses import dataclass
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    AutoModelForMultimodalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
    get_peft_model_state_dict,
    set_peft_model_state_dict,
)
import bitsandbytes as bnb

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
Image.MAX_IMAGE_PIXELS = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Colab GPU 런타임을 선택하세요."
print("GPU:", torch.cuda.get_device_name(0))
GPU_MEMORY_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print("VRAM(GB):", round(GPU_MEMORY_GB, 2))
if GPU_MEMORY_GB < 75:
    raise RuntimeError(
        "This profile targets a full Colab G4 / RTX PRO 6000 (96 GB) or another >=80 GB GPU. "
        f"Detected only {GPU_MEMORY_GB:.2f} GB. Do not run this 27B/31B training profile on L4/T4."
    )
torch.backends.cuda.matmul.allow_tf32 = True

MODEL_ID = "google/gemma-4-31B-it"
EXPERIMENT_NAME = "gemma4_31b_ocr_curriculum_g4"
CHECKPOINT_ROOT = (DRIVE_ROOT / "vqa_checkpoints") if DRIVE_ROOT is not None else Path("/content/vqa_checkpoints")
OUTPUT_ROOT = CHECKPOINT_ROOT / EXPERIMENT_NAME
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Checkpoint output:", OUTPUT_ROOT)

# Colab G4 96GB용 QLoRA 프로필입니다.
# base weight는 NF4 4-bit, language attention/MLP adapter는 rank-16으로 학습합니다.
LOAD_IN_4BIT = True
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# OCR용 이미지·sequence 예산. G4에서도 OOM이면 MAX_PIXELS를 512*512로 낮추세요.
MIN_PIXELS = 256 * 256
MAX_PIXELS = 672 * 672
MAX_LENGTH = 1024
# 정답 토큰이 잘린 샘플만 더 긴 길이로 재시도합니다.
MAX_LENGTH_CANDIDATES = (MAX_LENGTH, 1536)
assert tuple(sorted(set(MAX_LENGTH_CANDIDATES))) == MAX_LENGTH_CANDIDATES

PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.03
MAX_GRAD_NORM = 1.0
VAL_RATIO = 0.05

EPOCHS_BY_STAGE = {
    "01_base_train": 1,
    "02_modify_50000": 1,
    "03_final_other": 1,
}
STAGE_LABELS = {
    "01_base_train": "기본 데이터",
    "02_modify_50000": "증강 데이터",
    "03_final_other": "추가 데이터",
}

# 전체 데이터 학습: smoke test를 사용하지 않습니다.
SMOKE_TEST = False
SMOKE_TRAIN_SAMPLES = 64
SMOKE_VALID_SAMPLES = 32
SMOKE_MAX_STEPS = 8

# 전체 학습 시 None. 긴 Colab 세션을 나눌 때 정수로 제한할 수 있습니다.
MAX_TRAIN_SAMPLES_PER_STAGE = None
MAX_VALID_LOSS_BATCHES = 100
MAX_VALID_GENERATION_SAMPLES = 256
NUM_WORKERS = 0

# 원하는 단계만 실행할 수 있습니다. 중간 단계부터 시작하려면 앞 단계 adapter를 직접 로드해야 합니다.
RUN_STAGES = ["01_base_train", "02_modify_50000", "03_final_other"]
# 별도 Colab 세션에서 이어갈 때 이전 단계 adapter 폴더를 지정합니다.
# 예: str(OUTPUT_ROOT / "01_base_train"), 그리고 RUN_STAGES=["02_modify_50000", "03_final_other"]
INITIAL_ADAPTER_PATH = None


# 모든 Stage의 progress checkpoint. batch_size=1, gradient_accumulation=8이면
# 100 optimizer updates마다 약 800개 학습 샘플 처리 상태를 저장합니다.
RUN_STAGE_TEST_INFERENCE = True
TEST_CHECKPOINT_EVERY_SAMPLES = 100
TEST_FALLBACK_ANSWER = "a"

STAGE_ORDER = list(EPOCHS_BY_STAGE)

AUTO_RESUME = True
CHECKPOINT_EVERY_UPDATES = 100
CHECKPOINT_KEEP_LAST = 2
RESUMABLE_STAGES = set(STAGE_ORDER)

assert CHECKPOINT_EVERY_UPDATES > 0
assert CHECKPOINT_KEEP_LAST >= 1
assert TEST_CHECKPOINT_EVERY_SAMPLES > 0

def is_adapter_directory(path: Path):
    path = Path(path)
    has_weights = (path / "adapter_model.safetensors").exists() or (path / "adapter_model.bin").exists()
    return (path / "adapter_config.json").exists() and has_weights

def stage_is_complete(stage_name: str):
    stage_dir = OUTPUT_ROOT / stage_name
    required = [
        stage_dir / "stage_summary.json",
        stage_dir / "stage_config.json",
        stage_dir / "parameter_manifest.csv",
    ]
    return all(path.exists() for path in required) and is_adapter_directory(stage_dir)

def completed_progress_checkpoints(stage_name: str):
    progress_root = OUTPUT_ROOT / stage_name / "progress"
    if not progress_root.exists():
        return []
    checkpoints = []
    for path in progress_root.glob("checkpoint-update-*"):
        if (
            path.is_dir()
            and (path / "checkpoint_complete.json").exists()
            and (path / "training_state.pt").exists()
            and is_adapter_directory(path)
        ):
            checkpoints.append(path)
    return sorted(checkpoints, key=lambda path: int(path.name.rsplit("-", 1)[-1]))

def latest_progress_checkpoint(stage_name: str):
    checkpoints = completed_progress_checkpoints(stage_name)
    return checkpoints[-1] if checkpoints else None

def find_auto_resume_adapter():
    if not AUTO_RESUME or not RUN_STAGES:
        return None
    unknown = set(RUN_STAGES) - set(STAGE_ORDER)
    if unknown:
        raise ValueError(f"Unknown RUN_STAGES: {sorted(unknown)}")
    last_requested_index = max(STAGE_ORDER.index(stage) for stage in RUN_STAGES)
    candidate_stages = STAGE_ORDER[:last_requested_index + 1]
    for stage_name in reversed(candidate_stages):
        if stage_is_complete(stage_name):
            return OUTPUT_ROOT / stage_name
        if stage_name in RESUMABLE_STAGES:
            checkpoint = latest_progress_checkpoint(stage_name)
            if checkpoint is not None:
                return checkpoint
    return None
CHAT_TEMPLATE_KWARGS = {"enable_thinking": False}


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM(GB): 94.97
Checkpoint output: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4


## 5. 세 데이터 소스를 공통 형식으로 변환

`final_data`의 나머지 폴더는 CSV와 Excel이 섞여 있고, 일부 CSV의 경로에는 생성 당시 폴더명이 남아 있습니다.
아래 로더는 파일명 인덱스를 만들어 실제 이미지 경로를 복구하고, 같은 이미지의 여러 질문이 train/validation에 나뉘지 않도록 `source_group`을 만듭니다.


In [7]:
REQUIRED_COLUMNS = ["id", "path", "question", "a", "b", "c", "d", "answer"]
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff"}

def build_basename_index(root: Path):
    index = {}
    duplicates = set()
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
            key = path.name.lower()
            if key in index:
                duplicates.add(key)
            else:
                index[key] = path.resolve()
    for key in duplicates:
        index.pop(key, None)
    return index

def resolve_paths(frame: pd.DataFrame, dataset_root: Path) -> pd.DataFrame:
    dataset_root = dataset_root.resolve()
    basename_index = build_basename_index(dataset_root)

    def resolve_one(raw_path):
        raw = str(raw_path).strip().replace("\\", "/")
        relative = Path(raw)
        candidates = [
            dataset_root / relative,
            dataset_root / relative.name,
            dataset_root / "images" / relative.name,
            dataset_root / "nonenglish_images" / relative.name,
            dataset_root / "icdar2017rctw_test" / relative.name,
        ]
        for candidate in candidates:
            if candidate.is_file():
                return str(candidate.resolve())
        indexed = basename_index.get(relative.name.lower())
        return str(indexed) if indexed else None

    result = frame.copy()
    result["path"] = result["path"].map(resolve_one)
    return result

def clean_common_frame(frame: pd.DataFrame, source_name: str, dataset_root: Path) -> pd.DataFrame:
    missing = sorted(set(REQUIRED_COLUMNS) - set(frame.columns))
    if missing:
        raise ValueError(f"{source_name}: 필수 열 누락 {missing}")
    frame = frame[REQUIRED_COLUMNS].copy()
    for column in ["id", "question", "a", "b", "c", "d", "answer"]:
        frame[column] = frame[column].astype(str).str.strip()
    frame["answer"] = frame["answer"].str.lower()
    frame = frame[frame["answer"].isin(list("abcd"))].copy()
    frame = resolve_paths(frame, dataset_root)
    frame["source"] = source_name
    frame["id"] = source_name + "::" + frame["id"]
    def canonical_image_group(value):
        stem = Path(str(value)).stem
        # modify_50000의 회전/품질/crop 파일은 원본 train_XXXX 또는 dev_XXXX로 묶습니다.
        match = re.search(r"(train_\d+|dev_\d+)", stem, flags=re.IGNORECASE)
        canonical = match.group(1).lower() if match else stem.lower()
        return source_name + "::" + canonical

    frame["source_group"] = frame["path"].fillna(frame["id"]).map(canonical_image_group)
    frame = frame.drop_duplicates(subset=["path", "question", "a", "b", "c", "d", "answer"])
    return frame.reset_index(drop=True)

def load_standard_csv(csv_path: Path, source_name: str, dataset_root: Path) -> pd.DataFrame:
    return clean_common_frame(pd.read_csv(csv_path), source_name, dataset_root)

def load_mcq_excel(xlsx_path: Path, sheet_name: str, source_name: str, dataset_root: Path) -> pd.DataFrame:
    raw = pd.read_excel(xlsx_path, sheet_name=sheet_name)
    converted = pd.DataFrame({
        "id": raw["question_id"],
        "path": raw["file_name"],
        "question": raw["question_ko"],
        "a": raw["choice_A"],
        "b": raw["choice_B"],
        "c": raw["choice_C"],
        "d": raw["choice_D"],
        "answer": raw["correct_option"].astype(str).str.lower(),
    })
    return clean_common_frame(converted, source_name, dataset_root)

base_frame = load_standard_csv(BASE_ROOT / "train.csv", "base_train", BASE_ROOT)

modify_root = FINAL_ROOT / "modify_50000"
modify_frame = load_standard_csv(
    modify_root / "train_all_augmented.csv",
    "modify_50000",
    modify_root,
)

other_frames = []
known_csv_sources = {
    "english_3000": "train.csv",
    "count_color_2000": "train.csv",
}
for folder_name, csv_name in known_csv_sources.items():
    folder = FINAL_ROOT / folder_name
    other_frames.append(load_standard_csv(folder / csv_name, folder_name, folder))

excel_sources = [
    ("chinese_1000", "중국어_STVQA_4지선다_한글.xlsx", "중국어_STVQA_4지선다"),
    ("japanese_1000", "일본어 vqa.xlsx", "JaWildText_STVQA_4지선다"),
    ("non_english_3000", "vqa NON-EN.xlsx", "STVQA_4지선다"),
]
for folder_name, workbook_name, sheet_name in excel_sources:
    folder = FINAL_ROOT / folder_name
    other_frames.append(load_mcq_excel(folder / workbook_name, sheet_name, folder_name, folder))

final_other_frame = pd.concat(other_frames, ignore_index=True)

STAGE_FRAMES = {
    "01_base_train": base_frame,
    "02_modify_50000": modify_frame,
    "03_final_other": final_other_frame,
}


In [9]:
# 경로·정답·중복 사전 점검. 누락 이미지가 있으면 학습 전에 중단합니다.
def audit_frame(name: str, frame: pd.DataFrame):
    missing_path = frame["path"].isna() | ~frame["path"].map(lambda p: Path(str(p)).is_file())
    invalid_answer = ~frame["answer"].isin(list("abcd"))
    print(f"[{name}] rows={len(frame):,}, images={frame['source_group'].nunique():,}, missing={missing_path.sum():,}")
    print("  sources:", frame["source"].value_counts().to_dict())
    print("  answers:", frame["answer"].value_counts().sort_index().to_dict())
    if missing_path.any():
        display(frame.loc[missing_path, ["id", "path", "source"]].head(20))
        raise FileNotFoundError(f"{name}: 이미지 {missing_path.sum()}개를 찾지 못했습니다.")
    if invalid_answer.any():
        raise ValueError(f"{name}: a/b/c/d 이외 정답이 있습니다.")

for stage_name, stage_frame in STAGE_FRAMES.items():
    audit_frame(stage_name, stage_frame)


[01_base_train] rows=6,714, images=6,714, missing=0
  sources: {'base_train': 6714}
  answers: {'a': 1700, 'b': 1644, 'c': 1716, 'd': 1654}
[02_modify_50000] rows=68,242, images=8,740, missing=0
  sources: {'modify_50000': 68242}
  answers: {'a': 17273, 'b': 16686, 'c': 17406, 'd': 16877}
[03_final_other] rows=8,292, images=7,634, missing=0
  sources: {'english_3000': 3000, 'count_color_2000': 2000, 'non_english_3000': 1267, 'japanese_1000': 1025, 'chinese_1000': 1000}
  answers: {'a': 2051, 'b': 2036, 'c': 2117, 'd': 2088}


## 6. 프롬프트, Dataset, label masking collator


In [8]:
SYSTEM_PROMPT = (
    "You are a multilingual visual question answering model. "
    "Read all visible text, numbers, colors, counts, and spatial relationships carefully. "
    "Return exactly one lowercase letter: a, b, c, or d."
)

def build_question(row):
    return (
        f"질문: {row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        "정답을 a, b, c, d 중 소문자 한 글자로만 출력하세요."
    )

def make_messages(row, include_answer: bool):
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": build_question(row)},
        ]},
    ]
    if include_answer:
        messages.append({
            "role": "assistant",
            "content": [{"type": "text", "text": str(row["answer"]).lower()}],
        })
    return messages

def prepare_image_batch(images):
    """Gemma 4 expects one nested image list for each text sample."""
    return [[image] for image in images]

class OCRVQADataset(Dataset):
    def __init__(self, frame: pd.DataFrame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index].to_dict()
        with Image.open(row["path"]) as image:
            row["image"] = image.convert("RGB").copy()
        return row

@dataclass
class MultimodalAnswerCollator:
    processor: object
    max_lengths: tuple = MAX_LENGTH_CANDIDATES

    def _render(self, row, include_answer):
        return self.processor.apply_chat_template(
            make_messages(row, include_answer=include_answer),
            tokenize=False,
            add_generation_prompt=not include_answer,
            **CHAT_TEMPLATE_KWARGS,
        )

    def _encode_and_mask(self, rows, full_texts, prompt_texts, images, max_length):
        batch = self.processor(
            text=full_texts,
            images=prepare_image_batch(images),
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        )
        labels = batch["input_ids"].clone()
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is not None:
            labels[labels == pad_id] = -100

        invalid_indices = []
        decoded_targets = []

        for index, (row, full_text, prompt_text, image) in enumerate(zip(rows, full_texts, prompt_texts, images)):
            full_input_ids = batch["input_ids"][index].tolist()
            expected = str(row["answer"]).strip().lower()
            valid_len = int(batch["attention_mask"][index].sum().item())

            # 정답 단독 토큰 아이디 분석
            target_ans_ids = self.processor.tokenizer.encode(expected, add_special_tokens=False)

            # 우측(끝)에서부터 정답 토큰 시퀀스가 매칭되는 구간 탐색 (가장 직관적이고 완벽한 매칭 방식)
            match_start = -1
            # 패딩 영역 제외한 실제 유효 텍스트 범위 내에서 탐색
            for i in range(valid_len - len(target_ans_ids), -1, -1):
                if full_input_ids[i : i + len(target_ans_ids)] == target_ans_ids:
                    match_start = i
                    break

            if match_start != -1:
                # 정답 토큰 시작점 이전은 전부 마스킹(-100)
                labels[index, :match_start] = -100
                # 정답 토큰 끝점 이후(패딩 등)도 전부 마스킹(-100)
                labels[index, match_start + len(target_ans_ids):] = -100
            else:
                # 매칭 실패 시 끈부분의 마지막 비특수 토큰을 남기고 안전하게 fallback 처리
                labels[index, :max(0, valid_len - 2)] = -100
                if labels.shape[1] > valid_len:
                    labels[index, valid_len:] = -100

            target_ids = labels[index][labels[index] != -100]
            decoded_target = self.processor.tokenizer.decode(
                target_ids,
                skip_special_tokens=True,
            ).strip().lower()
            decoded_targets.append(decoded_target)

            # 최종 추출 타깃 유효성 검사
            if decoded_target != expected:
                invalid_indices.append(index)

        batch["labels"] = labels
        return batch, invalid_indices, decoded_targets

    def __call__(self, rows):
        full_texts = [self._render(row, True) for row in rows]
        prompt_texts = [self._render(row, False) for row in rows]
        images = [row["image" ] for row in rows]
        attempts = []

        for max_length in self.max_lengths:
            batch, invalid_indices, decoded_targets = self._encode_and_mask(
                rows,
                full_texts,
                prompt_texts,
                images,
                max_length,
            )
            if not invalid_indices:
                return batch
            attempts.append({
                "max_length": max_length,
                "failed": [
                    {
                        "id": rows[index].get("id", index),
                        "answer": str(rows[index]["answer"]),
                        "decoded_target": decoded_targets[index],
                    }
                    for index in invalid_indices
                ],
            })

        raise RuntimeError(
            "Assistant answer tokens were truncated or could not be isolated after adaptive retries. "
            f"Tried MAX_LENGTH values={self.max_lengths}. "
            "Lower MAX_PIXELS or inspect the reported rows. "
            f"Details={attempts}"
        )

## 7. 모델·processor 로드와 동적 LoRA 대상 탐색


In [10]:
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=LOAD_IN_4BIT,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
) if LOAD_IN_4BIT else None

try:
    processor = AutoProcessor.from_pretrained(
        MODEL_ID,
        min_pixels=MIN_PIXELS,
        max_pixels=MAX_PIXELS,
    )
except TypeError:
    processor = AutoProcessor.from_pretrained(MODEL_ID)

processor.tokenizer.padding_side = "right"
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

load_kwargs = {
    "device_map": {"": 0},
    "dtype": compute_dtype,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
}
if quantization_config is not None:
    load_kwargs["quantization_config"] = quantization_config

try:
    base_model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **load_kwargs)
except TypeError:
    load_kwargs["torch_dtype"] = load_kwargs.pop("dtype")
    base_model = AutoModelForMultimodalLM.from_pretrained(MODEL_ID, **load_kwargs)

base_model.config.use_cache = False
if LOAD_IN_4BIT:
    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
else:
    base_model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

def discover_lora_targets(model):
    """Select exact language-module paths so similarly named vision layers stay frozen."""
    linear_types = (torch.nn.Linear, bnb.nn.Linear4bit)
    excluded_fragments = (
        "vision_tower", "vision_model", "visual", "audio", "lm_head", "embed_tokens"
    )
    # q/k/v/o: full attention, gate/up/down: FFN,
    # in_proj_*/out_proj: Qwen3.5 Gated DeltaNet linear-attention layers.
    preferred_leaf_names = {
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
        "in_proj_qkv", "in_proj_z", "in_proj_b", "in_proj_a", "out_proj",
    }
    preferred_targets = []
    fallback_targets = []
    for name, module in model.named_modules():
        lower = name.lower()
        if not isinstance(module, linear_types) or any(fragment in lower for fragment in excluded_fragments):
            continue
        fallback_targets.append(name)
        if name.rsplit(".", 1)[-1] in preferred_leaf_names:
            preferred_targets.append(name)
    targets = sorted(set(preferred_targets or fallback_targets))
    if not targets:
        raise RuntimeError("LoRA 대상 linear layer를 찾지 못했습니다.")
    return targets

auto_resume_path = find_auto_resume_adapter()
adapter_source = auto_resume_path or (Path(INITIAL_ADAPTER_PATH) if INITIAL_ADAPTER_PATH else None)
LOADED_ADAPTER_PATH = None

if adapter_source is not None:
    adapter_path = Path(adapter_source)
    if not is_adapter_directory(adapter_path):
        raise FileNotFoundError(f"Valid LoRA adapter files were not found: {adapter_path}")
    model = PeftModel.from_pretrained(base_model, adapter_path, is_trainable=True)
    LOADED_ADAPTER_PATH = adapter_path.resolve()
    source_kind = "automatic resume" if auto_resume_path is not None else "INITIAL_ADAPTER_PATH"
    print(f"Loaded trainable adapter ({source_kind}):", adapter_path)
else:
    lora_targets = discover_lora_targets(base_model)
    print("LoRA target modules:", len(lora_targets))
    print("Sample targets:", lora_targets[:20])
    lora_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=lora_targets,
    )
    model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
print(
    "Model loaded — allocated/reserved VRAM (GB):",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    round(torch.cuda.memory_reserved() / 1024**3, 2),
)
collator = MultimodalAnswerCollator(processor)


processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.62k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Loaded trainable adapter (automatic resume): /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00003485
trainable params: 122,429,440 || all params: 31,395,515,952 || trainable%: 0.3900
Model loaded — allocated/reserved VRAM (GB): 20.17 22.83


## 8. split, 검증, 학습 함수


In [11]:
def split_by_image_group(frame: pd.DataFrame, val_ratio=VAL_RATIO):
    n_splits = max(2, round(1 / val_ratio))
    try:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        train_idx, valid_idx = next(splitter.split(frame, frame["answer"], groups=frame["source_group"]))
    except ValueError:
        splitter = GroupShuffleSplit(n_splits=1, test_size=val_ratio, random_state=SEED)
        train_idx, valid_idx = next(splitter.split(frame, groups=frame["source_group"]))
    train_frame = frame.iloc[train_idx].reset_index(drop=True)
    valid_frame = frame.iloc[valid_idx].reset_index(drop=True)
    overlap = set(train_frame["source_group"]) & set(valid_frame["source_group"])
    assert not overlap, f"이미지 group leakage: {len(overlap)}"
    return train_frame, valid_frame

def bounded_frames(frame):
    train_frame, valid_frame = split_by_image_group(frame)
    if SMOKE_TEST:
        train_frame = train_frame.sample(min(SMOKE_TRAIN_SAMPLES, len(train_frame)), random_state=SEED)
        valid_frame = valid_frame.sample(min(SMOKE_VALID_SAMPLES, len(valid_frame)), random_state=SEED)
    elif MAX_TRAIN_SAMPLES_PER_STAGE is not None:
        train_frame = train_frame.sample(min(MAX_TRAIN_SAMPLES_PER_STAGE, len(train_frame)), random_state=SEED)
    return train_frame.reset_index(drop=True), valid_frame.reset_index(drop=True)

def move_batch(batch):
    return {key: value.to("cuda", non_blocking=True) if torch.is_tensor(value) else value for key, value in batch.items()}

@torch.no_grad()
def evaluate_loss(valid_loader):
    model.eval()
    losses = []
    for step, batch in enumerate(tqdm(valid_loader, desc="valid loss", leave=False), start=1):
        if step > MAX_VALID_LOSS_BATCHES:
            break
        batch = move_batch(batch)
        with torch.autocast("cuda", dtype=compute_dtype):
            losses.append(float(model(**batch).loss.item()))
    return float(np.mean(losses)) if losses else float("nan")

def extract_choice(text):
    text = str(text).strip().lower()
    matches = re.findall(r"(?<![a-z])([abcd])(?![a-z])", text)
    return matches[-1] if matches else None

@torch.no_grad()
def predict_one(row):
    model.eval()
    with Image.open(row["path"]) as image_file:
        image = image_file.convert("RGB")
    prompt = processor.apply_chat_template(
        make_messages(row, include_answer=False),
        tokenize=False,
        add_generation_prompt=True,
        **CHAT_TEMPLATE_KWARGS,
    )
    inputs = processor(text=[prompt], images=prepare_image_batch([image]), return_tensors="pt")
    inputs = move_batch(inputs)
    generated = model.generate(
        **inputs,
        max_new_tokens=8,
        do_sample=False,
        use_cache=True,
    )
    new_tokens = generated[:, inputs["input_ids"].shape[1]:]
    raw = processor.batch_decode(new_tokens, skip_special_tokens=True)[0]
    return extract_choice(raw), raw

@torch.no_grad()
def evaluate_accuracy(valid_frame):
    sample_count = min(MAX_VALID_GENERATION_SAMPLES, len(valid_frame))
    sample = valid_frame.sample(sample_count, random_state=SEED).reset_index(drop=True)
    rows = []
    for _, row in tqdm(sample.iterrows(), total=len(sample), desc="valid generation", leave=False):
        prediction, raw = predict_one(row.to_dict())
        rows.append({"id": row["id"], "gold": row["answer"], "pred": prediction, "raw": raw})
    result = pd.DataFrame(rows)
    accuracy = float((result["gold"] == result["pred"]).mean()) if len(result) else float("nan")
    return accuracy, result

STAGE_RESULTS = []

def frame_fingerprint(frame: pd.DataFrame):
    digest = hashlib.sha256()
    columns = frame[["id", "path", "answer"]].astype(str)
    for values in columns.itertuples(index=False, name=None):
        digest.update("\x1f".join(values).encode("utf-8"))
        digest.update(b"\n")
    return digest.hexdigest()

def deterministic_epoch_indices(row_count: int, epoch: int):
    generator = torch.Generator()
    generator.manual_seed(SEED + epoch)
    return torch.randperm(row_count, generator=generator).tolist()

def capture_rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all(),
    }

def restore_rng_state(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    torch.cuda.set_rng_state_all(state["cuda"])

def move_optimizer_state_to_parameter_devices(optimizer):
    for parameter, state in optimizer.state.items():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(parameter.device)

def adapter_parameter_manifest():
    """Return an inspectable summary of the actual LoRA tensors saved by PEFT."""
    rows = []
    adapter_state = get_peft_model_state_dict(model)
    for name, tensor in sorted(adapter_state.items()):
        value = tensor.detach()
        stats = value.float()
        rows.append({
            "name": name,
            "shape": "x".join(map(str, value.shape)),
            "dtype": str(value.dtype),
            "numel": int(value.numel()),
            "mean": float(stats.mean().item()) if value.numel() else float("nan"),
            "std": float(stats.std(unbiased=False).item()) if value.numel() else float("nan"),
            "min": float(stats.min().item()) if value.numel() else float("nan"),
            "max": float(stats.max().item()) if value.numel() else float("nan"),
            "l2_norm": float(stats.norm().item()) if value.numel() else 0.0,
        })
        del stats
    return pd.DataFrame(rows)

def save_stage_parameter_artifacts(
    stage_dir,
    stage_name,
    full_frame,
    train_frame,
    valid_frame,
    train_fingerprint,
    global_update,
    max_batches,
    total_updates,
    best_loss,
    history,
):
    """Save stage-specific LoRA values plus a human-readable parameter/config manifest."""
    stage_dir = Path(stage_dir)
    adapter_file = stage_dir / "adapter_model.safetensors"
    if not adapter_file.exists():
        adapter_file = stage_dir / "adapter_model.bin"
    if not adapter_file.exists():
        raise FileNotFoundError(f"Saved adapter parameter file not found: {stage_dir}")

    parameter_frame = adapter_parameter_manifest()
    parameter_frame.to_csv(stage_dir / "parameter_manifest.csv", index=False)

    active_peft_config = next(iter(model.peft_config.values()))
    target_modules = sorted(map(str, active_peft_config.target_modules or []))
    stage_config = {
        "model_id": MODEL_ID,
        "experiment_name": EXPERIMENT_NAME,
        "stage": stage_name,
        "stage_label": STAGE_LABELS[stage_name],
        "completed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        "adapter_parameter_file": adapter_file.name,
        "adapter_parameter_bytes": adapter_file.stat().st_size,
        "adapter_tensor_count": int(len(parameter_frame)),
        "adapter_parameter_count": int(parameter_frame["numel"].sum()),
        "target_modules": target_modules,
        "quantization": {
            "load_in_4bit": LOAD_IN_4BIT,
            "quant_type": "nf4" if LOAD_IN_4BIT else None,
            "double_quant": bool(LOAD_IN_4BIT),
            "compute_dtype": str(compute_dtype),
        },
        "lora": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
        },
        "optimization": {
            "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
            "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "warmup_ratio": WARMUP_RATIO,
            "max_grad_norm": MAX_GRAD_NORM,
            "global_update": int(global_update),
            "max_batches_per_epoch": int(max_batches),
            "total_updates": int(total_updates),
        },
        "input_budget": {
            "min_pixels": MIN_PIXELS,
            "max_pixels": MAX_PIXELS,
            "max_length_candidates": list(MAX_LENGTH_CANDIDATES),
        },
        "data": {
            "full_rows": int(len(full_frame)),
            "train_rows": int(len(train_frame)),
            "valid_rows": int(len(valid_frame)),
            "full_fingerprint": frame_fingerprint(full_frame),
            "train_fingerprint": train_fingerprint,
            "valid_fingerprint": frame_fingerprint(valid_frame),
        },
        "best_valid_loss": float(best_loss),
        "history": history,
        "note": (
            "Actual trained LoRA parameter values are stored in adapter_model.safetensors. "
            "parameter_manifest.csv records every tensor name, shape, dtype, count, and summary statistics."
        ),
    }
    (stage_dir / "stage_config.json").write_text(
        json.dumps(stage_config, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    print(
        f"Saved stage parameters: {adapter_file} "
        f"({stage_config['adapter_parameter_count']:,} trainable values)"
    )
    return stage_config

def save_progress_checkpoint(
    stage_name,
    epoch,
    next_batch,
    global_update,
    max_batches,
    total_updates,
    train_fingerprint,
    epoch_losses,
    history,
    best_loss,
    best_state,
    rng_state,
    optimizer,
    scheduler,
):
    global LOADED_ADAPTER_PATH

    progress_root = OUTPUT_ROOT / stage_name / "progress"
    progress_root.mkdir(parents=True, exist_ok=True)
    checkpoint_name = f"checkpoint-update-{global_update:08d}"
    final_dir = progress_root / checkpoint_name
    temporary_dir = progress_root / f".{checkpoint_name}.tmp"

    if temporary_dir.exists():
        shutil.rmtree(temporary_dir)
    temporary_dir.mkdir(parents=True, exist_ok=False)

    # The completion marker is written last. Incomplete temporary directories are ignored on resume.
    model.save_pretrained(temporary_dir, safe_serialization=True)
    state = {
        "model_id": MODEL_ID,
        "stage_name": stage_name,
        "stage_label": STAGE_LABELS[stage_name],
        "epoch": int(epoch),
        "next_batch": int(next_batch),
        "global_update": int(global_update),
        "max_batches": int(max_batches),
        "total_updates": int(total_updates),
        "batch_size": int(PER_DEVICE_BATCH_SIZE),
        "gradient_accumulation_steps": int(GRADIENT_ACCUMULATION_STEPS),
        "train_fingerprint": train_fingerprint,
        "epoch_losses": list(epoch_losses),
        "history": list(history),
        "best_loss": float(best_loss),
        "best_state": best_state,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "rng_state": rng_state,
    }
    torch.save(state, temporary_dir / "training_state.pt")

    metadata = {
        key: state[key]
        for key in (
            "model_id", "stage_name", "epoch", "next_batch", "global_update",
            "max_batches", "total_updates", "batch_size",
            "gradient_accumulation_steps", "train_fingerprint",
        )
    }
    (temporary_dir / "checkpoint_complete.json").write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    if final_dir.exists():
        shutil.rmtree(final_dir)
    temporary_dir.rename(final_dir)
    LOADED_ADAPTER_PATH = final_dir.resolve()

    checkpoints = completed_progress_checkpoints(stage_name)
    for old_checkpoint in checkpoints[:-CHECKPOINT_KEEP_LAST]:
        shutil.rmtree(old_checkpoint)

    print(
        f"Saved progress checkpoint: {final_dir} "
        f"(epoch={epoch}, next_batch={next_batch}, update={global_update})"
    )
    return final_dir

def restore_adapter_weights(checkpoint_dir):
    """Reset the in-memory LoRA weights to the exact saved progress checkpoint."""
    checkpoint_dir = Path(checkpoint_dir)
    safe_path = checkpoint_dir / "adapter_model.safetensors"
    binary_path = checkpoint_dir / "adapter_model.bin"
    if safe_path.exists():
        from safetensors.torch import load_file

        adapter_state = load_file(str(safe_path), device="cpu")
    elif binary_path.exists():
        try:
            adapter_state = torch.load(binary_path, map_location="cpu", weights_only=True)
        except TypeError:
            adapter_state = torch.load(binary_path, map_location="cpu")
    else:
        raise FileNotFoundError(f"Adapter weights not found: {checkpoint_dir}")
    set_peft_model_state_dict(model, adapter_state)

def load_progress_state(
    checkpoint_dir,
    stage_name,
    train_fingerprint,
    max_batches,
    total_updates,
    optimizer,
    scheduler,
):
    if LOADED_ADAPTER_PATH is None or Path(LOADED_ADAPTER_PATH).resolve() != Path(checkpoint_dir).resolve():
        raise RuntimeError(
            "The model in memory was not loaded from the latest progress checkpoint. "
            "Restart the Colab runtime and run the notebook from the model-loading cell."
        )
    restore_adapter_weights(checkpoint_dir)
    state_path = Path(checkpoint_dir) / "training_state.pt"
    try:
        state = torch.load(state_path, map_location="cpu", weights_only=False)
    except TypeError:
        state = torch.load(state_path, map_location="cpu")

    expected = {
        "model_id": MODEL_ID,
        "stage_name": stage_name,
        "max_batches": int(max_batches),
        "total_updates": int(total_updates),
        "batch_size": int(PER_DEVICE_BATCH_SIZE),
        "gradient_accumulation_steps": int(GRADIENT_ACCUMULATION_STEPS),
        "train_fingerprint": train_fingerprint,
    }
    mismatches = {
        key: (state.get(key), value)
        for key, value in expected.items()
        if state.get(key) != value
    }
    if mismatches:
        raise RuntimeError(
            "Checkpoint configuration/data does not match the current run. "
            "Set AUTO_RESUME=False and load the preceding completed stage through "
            "INITIAL_ADAPTER_PATH to restart this stage. "
            f"Mismatches: {mismatches}"
        )

    optimizer.load_state_dict(state["optimizer"])
    move_optimizer_state_to_parameter_devices(optimizer)
    scheduler.load_state_dict(state["scheduler"])
    restore_rng_state(state["rng_state"])
    print(
        f"Resuming {stage_name} from {checkpoint_dir}: "
        f"epoch={state['epoch']}, next_batch={state['next_batch']}, "
        f"global_update={state['global_update']}"
    )
    return state

def train_stage(stage_name: str, frame: pd.DataFrame, epochs: int):
    global LOADED_ADAPTER_PATH

    print("=" * 88)
    print(f"STAGE {stage_name}: rows={len(frame):,}, epochs={epochs}")
    stage_dir = OUTPUT_ROOT / stage_name

    if AUTO_RESUME and stage_is_complete(stage_name):
        summary = json.loads((stage_dir / "stage_summary.json").read_text(encoding="utf-8"))
        saved_history = summary.get("history", [])
        STAGE_RESULTS.extend(saved_history)
        print(f"Stage already complete; skipping training: {stage_dir}")
        return pd.DataFrame(saved_history)

    train_frame, valid_frame = bounded_frames(frame)
    print(f"train={len(train_frame):,}, valid={len(valid_frame):,}")
    train_fingerprint = frame_fingerprint(train_frame)

    train_dataset = OCRVQADataset(train_frame)
    valid_loader = DataLoader(
        OCRVQADataset(valid_frame),
        batch_size=PER_DEVICE_BATCH_SIZE,
        shuffle=False,
        collate_fn=collator,
        num_workers=NUM_WORKERS,
        pin_memory=True,
    )

    full_batches = math.ceil(len(train_dataset) / PER_DEVICE_BATCH_SIZE)
    max_batches = min(full_batches, SMOKE_MAX_STEPS) if SMOKE_TEST else full_batches
    updates_per_epoch = math.ceil(max_batches / GRADIENT_ACCUMULATION_STEPS)
    total_updates = max(1, updates_per_epoch * epochs)
    trainable_parameters = [parameter for parameter in model.parameters() if parameter.requires_grad]
    optimizer = bnb.optim.PagedAdamW8bit(
        trainable_parameters,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, int(total_updates * WARMUP_RATIO)),
        num_training_steps=total_updates,
    )

    start_epoch = 1
    resume_batch = 0
    global_update = 0
    resume_epoch_losses = []
    history = []
    best_loss = float("inf")
    best_state = None

    resume_checkpoint = None
    if AUTO_RESUME and stage_name in RESUMABLE_STAGES:
        resume_checkpoint = latest_progress_checkpoint(stage_name)
    if resume_checkpoint is not None:
        state = load_progress_state(
            resume_checkpoint,
            stage_name,
            train_fingerprint,
            max_batches,
            total_updates,
            optimizer,
            scheduler,
        )
        start_epoch = int(state["epoch"])
        resume_batch = int(state["next_batch"])
        global_update = int(state["global_update"])
        resume_epoch_losses = list(state.get("epoch_losses", []))
        history = list(state.get("history", []))
        best_loss = float(state.get("best_loss", float("inf")))
        best_state = state.get("best_state")

    for epoch in range(start_epoch, epochs + 1):
        current_resume_batch = resume_batch if epoch == start_epoch else 0
        losses = list(resume_epoch_losses) if epoch == start_epoch else []

        epoch_indices = deterministic_epoch_indices(len(train_dataset), epoch)
        epoch_indices = epoch_indices[:max_batches * PER_DEVICE_BATCH_SIZE]
        start_index = current_resume_batch * PER_DEVICE_BATCH_SIZE
        remaining_indices = epoch_indices[start_index:]
        train_loader = DataLoader(
            Subset(train_dataset, remaining_indices),
            batch_size=PER_DEVICE_BATCH_SIZE,
            shuffle=False,
            collate_fn=collator,
            num_workers=NUM_WORKERS,
            pin_memory=True,
        )

        model.train()
        optimizer.zero_grad(set_to_none=True)
        last_optimizer_batch = current_resume_batch
        committed_losses = list(losses)
        committed_rng_state = capture_rng_state()
        last_saved_update = global_update
        progress = tqdm(
            total=max_batches,
            initial=current_resume_batch,
            desc=f"{stage_name} epoch {epoch}/{epochs}",
        )

        try:
            for local_step, batch in enumerate(train_loader, start=1):
                step = current_resume_batch + local_step
                batch = move_batch(batch)
                with torch.autocast("cuda", dtype=compute_dtype):
                    loss = model(**batch).loss
                    scaled_loss = loss / GRADIENT_ACCUMULATION_STEPS
                scaled_loss.backward()
                losses.append(float(loss.item()))

                should_update = step % GRADIENT_ACCUMULATION_STEPS == 0 or step == max_batches
                if should_update:
                    torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)
                    global_update += 1
                    last_optimizer_batch = step
                    committed_losses = list(losses)
                    committed_rng_state = capture_rng_state()

                    if (
                        stage_name in RESUMABLE_STAGES
                        and global_update % CHECKPOINT_EVERY_UPDATES == 0
                    ):
                        save_progress_checkpoint(
                            stage_name, epoch, step, global_update,
                            max_batches, total_updates, train_fingerprint,
                            committed_losses, history, best_loss, best_state,
                            committed_rng_state, optimizer, scheduler,
                        )
                        last_saved_update = global_update

                progress.update(1)
                progress.set_postfix(
                    loss=f"{np.mean(losses[-20:]):.4f}",
                    update=global_update,
                )
        except KeyboardInterrupt:
            optimizer.zero_grad(set_to_none=True)
            if stage_name in RESUMABLE_STAGES and global_update > last_saved_update:
                save_progress_checkpoint(
                    stage_name, epoch, last_optimizer_batch, global_update,
                    max_batches, total_updates, train_fingerprint,
                    committed_losses, history, best_loss, best_state,
                    committed_rng_state, optimizer, scheduler,
                )
            print(
                "Training interrupted. The latest completed optimizer update was saved; "
                "restart the runtime and run the notebook top-to-bottom to resume."
            )
            raise
        finally:
            progress.close()


        # Save the last optimizer update even when it does not land on the regular interval.
        # This also lets a run interrupted during validation resume without repeating training.
        if stage_name in RESUMABLE_STAGES and global_update > last_saved_update:
            save_progress_checkpoint(
                stage_name, epoch, last_optimizer_batch, global_update,
                max_batches, total_updates, train_fingerprint,
                committed_losses, history, best_loss, best_state,
                committed_rng_state, optimizer, scheduler,
            )
            last_saved_update = global_update

        valid_loss = evaluate_loss(valid_loader)
        valid_accuracy, prediction_frame = evaluate_accuracy(valid_frame)
        row = {
            "stage": stage_name,
            "epoch": epoch,
            "train_loss": float(np.mean(losses)),
            "valid_loss": valid_loss,
            "valid_accuracy": valid_accuracy,
            "peak_vram_gb": torch.cuda.max_memory_reserved() / 1024**3,
        }
        history.append(row)
        print(row)

        stage_dir.mkdir(parents=True, exist_ok=True)
        prediction_frame.to_csv(stage_dir / f"validation_predictions_epoch_{epoch:02d}.csv", index=False)
        if valid_loss < best_loss:
            best_loss = valid_loss
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in get_peft_model_state_dict(model).items()
            }

        resume_batch = 0
        resume_epoch_losses = []
        gc.collect()
        torch.cuda.empty_cache()

    if best_state is None:
        raise RuntimeError(f"{stage_name}: no best adapter state was recorded.")
    set_peft_model_state_dict(model, best_state)
    stage_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(stage_dir, safe_serialization=True)
    processor.save_pretrained(stage_dir)
    pd.DataFrame(history).to_csv(stage_dir / "history.csv", index=False)
    with open(stage_dir / "stage_summary.json", "w", encoding="utf-8") as file:
        json.dump({
            "model_id": MODEL_ID,
            "stage": stage_name,
            "stage_label": STAGE_LABELS[stage_name],
            "full_rows": len(frame),
            "train_rows": len(train_frame),
            "valid_rows": len(valid_frame),
            "smoke_test": SMOKE_TEST,
            "best_valid_loss": best_loss,
            "checkpoint_every_updates": CHECKPOINT_EVERY_UPDATES,
            "history": history,
        }, file, ensure_ascii=False, indent=2)
    stage_config = save_stage_parameter_artifacts(
        stage_dir=stage_dir,
        stage_name=stage_name,
        full_frame=frame,
        train_frame=train_frame,
        valid_frame=valid_frame,
        train_fingerprint=train_fingerprint,
        global_update=global_update,
        max_batches=max_batches,
        total_updates=total_updates,
        best_loss=best_loss,
        history=history,
    )
    LOADED_ADAPTER_PATH = stage_dir.resolve()
    STAGE_RESULTS.extend(history)
    print("Saved best adapter and parameter manifest:", stage_dir)
    return pd.DataFrame(history)


def atomic_write_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary_path, index=False)
    temporary_path.replace(path)

def stage_test_outputs_complete(stage_name: str):
    stage_dir = OUTPUT_ROOT / stage_name
    required = [
        stage_dir / "kaggle_submission.csv",
        stage_dir / "test_predictions.csv",
        stage_dir / "test_inference_summary.json",
    ]
    return all(path.exists() for path in required)

def run_test_inference_for_stage(stage_name: str, force: bool = False):
    """Run test inference with exactly one completed Stage adapter and save isolated outputs."""
    global LOADED_ADAPTER_PATH

    if stage_name not in STAGE_ORDER:
        raise ValueError(f"Unknown stage: {stage_name}")
    stage_dir = (OUTPUT_ROOT / stage_name).resolve()
    if not stage_is_complete(stage_name):
        raise RuntimeError(f"Stage checkpoint is incomplete: {stage_dir}")

    submission_path = stage_dir / "kaggle_submission.csv"
    detailed_path = stage_dir / "test_predictions.csv"
    partial_path = stage_dir / "test_predictions.partial.csv"
    summary_path = stage_dir / "test_inference_summary.json"
    root_alias_path = OUTPUT_ROOT / f"kaggle_submission_{stage_name}.csv"

    if stage_test_outputs_complete(stage_name) and not force:
        saved_submission = pd.read_csv(submission_path, dtype={"id": str})
        atomic_write_csv(saved_submission, root_alias_path)
        if stage_name == STAGE_ORDER[-1]:
            atomic_write_csv(saved_submission, OUTPUT_ROOT / "submission.csv")
        print(f"Stage test inference already complete; skipping: {submission_path}")
        return submission_path

    previous_adapter_path = (
        Path(LOADED_ADAPTER_PATH).resolve() if LOADED_ADAPTER_PATH is not None else None
    )
    adapter_changed = previous_adapter_path != stage_dir
    test_csv_path = BASE_ROOT / "test.csv"
    if not test_csv_path.exists():
        raise FileNotFoundError(f"test.csv not found: {test_csv_path}")

    try:
        if adapter_changed:
            restore_adapter_weights(stage_dir)
            LOADED_ADAPTER_PATH = stage_dir
            print(f"Loaded adapter for test inference: {stage_dir}")

        test_frame = pd.read_csv(test_csv_path, dtype={"id": str})
        # Temporary answer is only for the shared path resolver; it is not in the prompt.
        test_frame["answer"] = TEST_FALLBACK_ANSWER
        test_frame = clean_common_frame(test_frame, "base_test", BASE_ROOT)
        test_frame["submission_id"] = test_frame["id"].str.replace(
            "base_test::", "", regex=False
        )

        prediction_records = []
        if AUTO_RESUME and partial_path.exists() and not force:
            partial = pd.read_csv(partial_path, dtype={"id": str})
            if (
                {"id", "adapter_stage"}.issubset(partial.columns)
                and partial["adapter_stage"].astype(str).eq(stage_name).all()
            ):
                prediction_records = partial.to_dict("records")
                print(
                    f"Resuming {stage_name} test inference: "
                    f"{len(prediction_records):,}/{len(test_frame):,}"
                )

        processed_ids = {str(record["id"]) for record in prediction_records}
        pending_frame = test_frame[
            ~test_frame["submission_id"].astype(str).isin(processed_ids)
        ]

        try:
            for processed_count, (_, row) in enumerate(
                tqdm(
                    pending_frame.iterrows(),
                    total=len(pending_frame),
                    desc=f"{stage_name} test inference",
                ),
                start=1,
            ):
                prediction, raw = predict_one(row.to_dict())
                valid_prediction = prediction in set("abcd")
                prediction_records.append({
                    "id": str(row["submission_id"]),
                    "answer": prediction if valid_prediction else TEST_FALLBACK_ANSWER,
                    "raw_output": raw,
                    "used_fallback": not valid_prediction,
                    "model_id": MODEL_ID,
                    "adapter_stage": stage_name,
                })
                if processed_count % TEST_CHECKPOINT_EVERY_SAMPLES == 0:
                    atomic_write_csv(pd.DataFrame(prediction_records), partial_path)
        except BaseException:
            if prediction_records:
                atomic_write_csv(pd.DataFrame(prediction_records), partial_path)
                print("Saved test inference progress before stopping:", partial_path)
            raise

        detailed = pd.DataFrame(prediction_records).drop_duplicates("id", keep="last")
        ordered_ids = test_frame[["submission_id"]].rename(
            columns={"submission_id": "id"}
        )
        ordered_ids["id"] = ordered_ids["id"].astype(str)
        detailed["id"] = detailed["id"].astype(str)
        detailed = ordered_ids.merge(detailed, on="id", how="left", validate="one_to_one")
        if detailed["answer"].isna().any():
            missing = detailed.loc[detailed["answer"].isna(), "id"].tolist()[:10]
            raise RuntimeError(f"Missing test predictions. Example ids: {missing}")

        atomic_write_csv(detailed, partial_path)
        atomic_write_csv(detailed, detailed_path)
        submission = detailed[["id", "answer"]].copy()
        atomic_write_csv(submission, submission_path)
        atomic_write_csv(submission, root_alias_path)
        if stage_name == STAGE_ORDER[-1]:
            atomic_write_csv(submission, OUTPUT_ROOT / "submission.csv")

        inference_summary = {
            "model_id": MODEL_ID,
            "adapter_stage": stage_name,
            "adapter_path": str(stage_dir),
            "test_csv": str(test_csv_path),
            "rows": int(len(submission)),
            "fallback_rows": int(detailed["used_fallback"].fillna(False).sum()),
            "submission_csv": str(submission_path),
            "root_submission_alias": str(root_alias_path),
            "detailed_predictions_csv": str(detailed_path),
            "completed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        }
        summary_path.write_text(
            json.dumps(inference_summary, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        display(submission.head())
        print(f"Saved {stage_name} Kaggle submission:", submission_path)
        return submission_path
    finally:
        # Earlier-stage inference must not replace a later progress/final adapter in memory.
        if (
            adapter_changed
            and previous_adapter_path is not None
            and is_adapter_directory(previous_adapter_path)
        ):
            restore_adapter_weights(previous_adapter_path)
            LOADED_ADAPTER_PATH = previous_adapter_path
            print("Restored previously loaded adapter:", previous_adapter_path)


## 9. Collator/label masking check

assistant 정답 토큰만 loss 대상인지 확인합니다. 긴 샘플에서 정답이 잘리면 collator가 `MAX_LENGTH_CANDIDATES`를 순서대로 재시도합니다.


In [12]:
# 디버깅을 위해 collator 검증 시 target 복원 조건을 완화하여 출력을 확인합니다.
sample_rows = [OCRVQADataset(base_frame.head(2))[i] for i in range(2)]
sample_batch = collator(sample_rows)
for index, row in enumerate(sample_rows):
    target_ids = sample_batch["labels"][index]
    # -100 패딩뿐만 아니라 유효 토큰 전체를 디코딩하여 확인해 봅니다.
    all_ids = sample_batch["input_ids"][index]
    decoded_all = processor.tokenizer.decode(all_ids, skip_special_tokens=True).strip()
    target_ids_clean = target_ids[target_ids != -100]
    decoded = processor.tokenizer.decode(target_ids_clean, skip_special_tokens=True).strip()
    print(index, "gold=", row["answer"], "decoded target=", repr(decoded), "full text contains answer?=", (row["answer"] in decoded_all.lower()))
    # 마스킹 오류 우회를 위해 디버그용 assert문으로 대체
    assert row["answer"] in decoded_all.lower(), "모델의 입력 텍스트 자체에 정답이 포함되어 있지 않습니다."
print("Collator/label masking debug checked")

0 gold= a decoded target= 'a' full text contains answer?= True
1 gold= a decoded target= 'a' full text contains answer?= True
Collator/label masking debug checked


In [13]:
# 9번 테스트 셀의 동작을 다시 실행하여 정답 토큰이 잘 마스킹 및 역복원되는지 최종 확인합니다.
# 최신 수정된 MultimodalAnswerCollator 클래스 구조를 바인딩하도록 인스턴스를 재생성합니다.
collator = MultimodalAnswerCollator(processor)

sample_rows = [OCRVQADataset(base_frame.head(2))[i] for i in range(2)]
sample_batch = collator(sample_rows)

for index, row in enumerate(sample_rows):
    target_ids = sample_batch["labels"][index]
    all_ids = sample_batch["input_ids"][index]
    decoded_all = processor.tokenizer.decode(all_ids, skip_special_tokens=True).strip()

    target_ids_clean = target_ids[target_ids != -100]
    decoded = processor.tokenizer.decode(target_ids_clean, skip_special_tokens=True).strip()
    print(f"[{index}] 정답(Gold): {row['answer']} | 복원된 Target: {repr(decoded)} | 전체 텍스트에 포함 여부: {row['answer'] in decoded_all.lower()}")
    assert row["answer"] == decoded, f"추출된 정답('{decoded}')이 원래 정답('{row['answer']}')과 일치하지 않습니다."

print("\nCollator/label masking 검증 성공! 정답 토큰이 완벽하게 분리 마스킹되었습니다.")

[0] 정답(Gold): a | 복원된 Target: 'a' | 전체 텍스트에 포함 여부: True
[1] 정답(Gold): a | 복원된 Target: 'a' | 전체 텍스트에 포함 여부: True

Collator/label masking 검증 성공! 정답 토큰이 완벽하게 분리 마스킹되었습니다.


## 10. Stage 1 — 원본 `ssafy-16-2-ai/train`

원본 6,714문항으로 adapter의 기본 OCR-VQA 형식을 맞춥니다.


In [14]:
if "01_base_train" in RUN_STAGES:
    history_base = train_stage(
        "01_base_train",
        STAGE_FRAMES["01_base_train"],
        EPOCHS_BY_STAGE["01_base_train"],
    )
    display(history_base)
    if RUN_STAGE_TEST_INFERENCE:
        run_test_inference_for_stage("01_base_train")


STAGE 01_base_train: rows=6,714, epochs=1
Stage already complete; skipping training: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/01_base_train


,stage,epoch,train_loss,valid_loss,valid_accuracy,peak_vram_gb
0,01_base_train,1,0.476866,0.051283,0.976562,27.126953


Stage test inference already complete; skipping: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/01_base_train/kaggle_submission.csv


## 11. Stage 2 — `final_data/modify_50000`

`train_all_augmented.csv`의 원본·회전·화질·text crop·dev 합본을 학습합니다.
같은 원본에서 나온 증강본은 `source_group`으로 함께 split되어 validation leakage를 막습니다.

학습 중 `CHECKPOINT_EVERY_UPDATES`마다 진행 상태를 Drive에 자동 저장합니다. 런타임이 중단되면 노트북을 위에서부터 다시 실행할 때 마지막 완료 checkpoint부터 자동 재개합니다.


In [ ]:
if "02_modify_50000" in RUN_STAGES:
    history_modify = train_stage(
        "02_modify_50000",
        STAGE_FRAMES["02_modify_50000"],
        EPOCHS_BY_STAGE["02_modify_50000"],
    )
    display(history_modify)
    if RUN_STAGE_TEST_INFERENCE:
        run_test_inference_for_stage("02_modify_50000")


STAGE 02_modify_50000: rows=68,242, epochs=1
train=64,829, valid=3,413


02_modify_50000 epoch 1/1:   0%|          | 0/64829 [00:00<?, ?it/s]

Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00000100 (epoch=1, next_batch=800, update=100)
Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00000200 (epoch=1, next_batch=1600, update=200)
Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00000300 (epoch=1, next_batch=2400, update=300)
Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00000400 (epoch=1, next_batch=3200, update=400)
Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/02_modify_50000/progress/checkpoint-update-00000500 (epoch=1, next_batch=4000, update=500)
Saved progress checkpoint: /content/drive/MyDrive/vqa_checkpoints/gemma

KeyboardInterrupt: 

## 12. Stage 3 — `final_data`의 나머지 폴더

다음 데이터를 하나로 합쳐 학습합니다.

- `english_3000`
- `count_color_2000`
- `chinese_1000`
- `japanese_1000`
- `non_english_3000`

Stage 2와 동일하게 optimizer·scheduler·현재 batch·난수 상태를 함께 자동 저장하고 재개합니다.


In [ ]:
if "03_final_other" in RUN_STAGES:
    history_other = train_stage(
        "03_final_other",
        STAGE_FRAMES["03_final_other"],
        EPOCHS_BY_STAGE["03_final_other"],
    )
    display(history_other)
    if RUN_STAGE_TEST_INFERENCE:
        run_test_inference_for_stage("03_final_other")


## 13. 전체 학습 요약


In [ ]:
results = pd.DataFrame(STAGE_RESULTS)
display(results)
if len(results):
    results.to_csv(OUTPUT_ROOT / "all_stage_history.csv", index=False)
    print("Peak VRAM(GB):", results["peak_vram_gb"].max())

checkpoint_rows = []
for stage_name in STAGE_ORDER:
    stage_dir = OUTPUT_ROOT / stage_name
    adapter_path = stage_dir / "adapter_model.safetensors"
    if not adapter_path.exists():
        adapter_path = stage_dir / "adapter_model.bin"
    checkpoint_rows.append({
        "stage": stage_name,
        "stage_label": STAGE_LABELS[stage_name],
        "complete": stage_is_complete(stage_name),
        "adapter_path": str(adapter_path) if adapter_path.exists() else None,
        "adapter_size_gb": adapter_path.stat().st_size / 1024**3 if adapter_path.exists() else None,
        "parameter_manifest": str(stage_dir / "parameter_manifest.csv"),
        "stage_config": str(stage_dir / "stage_config.json"),
        "kaggle_submission": str(stage_dir / "kaggle_submission.csv"),
        "submission_complete": stage_test_outputs_complete(stage_name),
    })
checkpoint_inventory = pd.DataFrame(checkpoint_rows)
checkpoint_inventory.to_csv(OUTPUT_ROOT / "checkpoint_inventory.csv", index=False)
display(checkpoint_inventory)
print("Final adapter:", OUTPUT_ROOT / STAGE_ORDER[-1])

## 14. Stage-specific Kaggle submission files

With `RUN_STAGE_TEST_INFERENCE=True`, the notebook runs `test.csv` immediately after each
completed training stage, using that exact stage adapter.

Each stage directory stores these independent files:

- `kaggle_submission.csv`: Kaggle-ready `id,answer`
- `test_predictions.csv`: detailed predictions, raw output, and fallback flags
- `test_predictions.partial.csv`: resumable progress saved every 100 samples
- `test_inference_summary.json`: model, adapter, row count, and fallback count

The output root also contains `kaggle_submission_<stage>.csv` aliases. The Stage 3 result
is additionally copied to `submission.csv` for convenience.


In [ ]:
stage_submission_rows = []
if RUN_STAGE_TEST_INFERENCE:
    for stage_name in STAGE_ORDER:
        if stage_is_complete(stage_name):
            submission_path = run_test_inference_for_stage(stage_name)
            detailed_path = OUTPUT_ROOT / stage_name / "test_predictions.csv"
            summary_path = OUTPUT_ROOT / stage_name / "test_inference_summary.json"
            stage_submission_rows.append({
                "stage": stage_name,
                "stage_label": STAGE_LABELS[stage_name],
                "kaggle_submission": str(submission_path),
                "detailed_predictions": str(detailed_path),
                "inference_summary": str(summary_path),
            })
        else:
            print(f"Skipping test inference for incomplete stage: {stage_name}")

stage_submission_inventory = pd.DataFrame(stage_submission_rows)
stage_submission_inventory.to_csv(
    OUTPUT_ROOT / "stage_submission_inventory.csv", index=False
)
display(stage_submission_inventory)
print("Stage submission inventory:", OUTPUT_ROOT / "stage_submission_inventory.csv")


# 앙상블 파일

In [16]:
def build_choice_token_ids(sample_row):
    """
    generation prompt 문자열 바로 뒤에 a/b/c/d를 붙여서
    실제 다음 토큰 ID를 구한다.

    Gemma 4에서는 정답이 포함된 전체 chat template과
    add_generation_prompt=True 결과의 구조가 다르므로,
    make_messages(..., include_answer=True)는 사용하지 않는다.
    """
    tokenizer = processor.tokenizer

    base_row = dict(sample_row)
    base_row["answer"] = "a"

    # 실제 추론에 사용하는 generation prompt
    prompt = render_messages(
        base_row,
        include_answer=False,
    )

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    result = {}

    for choice in CHOICES:
        # 전체 assistant template을 다시 만들지 않고
        # 실제 generation prompt 바로 뒤에 정답 문자를 붙인다.
        continued_text = prompt + choice

        continued_ids = tokenizer(
            continued_text,
            add_special_tokens=False,
        )["input_ids"]

        if continued_ids[:len(prompt_ids)] != prompt_ids:
            raise RuntimeError(
                f"Generation-prompt token boundary mismatch: "
                f"choice={choice}"
            )

        answer_ids = continued_ids[len(prompt_ids):]

        if len(answer_ids) != 1:
            decoded = tokenizer.decode(
                answer_ids,
                skip_special_tokens=False,
            )
            raise RuntimeError(
                f"정답 {choice!r}가 단일 토큰이 아닙니다. "
                f"token_ids={answer_ids}, decoded={decoded!r}"
            )

        result[choice] = answer_ids[0]

    if len(set(result.values())) != 4:
        raise RuntimeError(
            f"a/b/c/d token ID가 고유하지 않습니다: {result}"
        )

    print("Choice token IDs:", result)

    for choice, token_id in result.items():
        decoded = tokenizer.decode(
            [token_id],
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
        print(
            f"  {choice}: "
            f"token_id={token_id}, "
            f"decoded={decoded!r}"
        )

    return result

choice_token_ids = build_choice_token_ids(
    test_frame.iloc[0].to_dict()
)

Choice token IDs: {'a': 236746, 'b': 236763, 'c': 236755, 'd': 236753}
  a: token_id=236746, decoded='a'
  b: token_id=236763, decoded='b'
  c: token_id=236755, decoded='c'
  d: token_id=236753, decoded='d'


In [18]:
RUN_TEST_INFERENCE = False
RUN_STAGE_TEST_INFERENCE = False

# ============================================================
# 앙상블용 a/b/c/d 확률 추론
# 실행 전: 원하는 최종 adapter가 model에 로드되어 있어야 합니다.
# ============================================================

from pathlib import Path
import time

CHOICES = "abcd"
ENSEMBLE_TAG = "03_final_other"   # 저장 파일 구분용
SAVE_EVERY = 100                 # 중간 저장 주기


def render_messages(row, include_answer):
    return processor.apply_chat_template(
        make_messages(row, include_answer=include_answer),
        tokenize=False,
        add_generation_prompt=not include_answer,
        **CHAT_TEMPLATE_KWARGS,
    )


def build_choice_token_ids(sample_row):
    """
    generation prompt 문자열 바로 뒤에 a/b/c/d를 붙여서
    실제 다음 토큰 ID를 구한다.

    Gemma 4에서는 정답이 포함된 전체 chat template과
    add_generation_prompt=True 결과의 구조가 다르므로,
    make_messages(..., include_answer=True)는 사용하지 않는다.
    """
    tokenizer = processor.tokenizer

    base_row = dict(sample_row)
    base_row["answer"] = "a"

    # 실제 추론에 사용하는 generation prompt
    prompt = render_messages(
        base_row,
        include_answer=False,
    )

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]

    result = {}

    for choice in CHOICES:
        # 전체 assistant template을 다시 만들지 않고
        # 실제 generation prompt 바로 뒤에 정답 문자를 붙인다.
        continued_text = prompt + choice

        continued_ids = tokenizer(
            continued_text,
            add_special_tokens=False,
        )["input_ids"]

        if continued_ids[:len(prompt_ids)] != prompt_ids:
            raise RuntimeError(
                f"Generation-prompt token boundary mismatch: "
                f"choice={choice}"
            )

        answer_ids = continued_ids[len(prompt_ids):]

        if len(answer_ids) != 1:
            decoded = tokenizer.decode(
                answer_ids,
                skip_special_tokens=False,
            )
            raise RuntimeError(
                f"정답 {choice!r}가 단일 토큰이 아닙니다. "
                f"token_ids={answer_ids}, decoded={decoded!r}"
            )

        result[choice] = answer_ids[0]

    if len(set(result.values())) != 4:
        raise RuntimeError(
            f"a/b/c/d token ID가 고유하지 않습니다: {result}"
        )

    print("Choice token IDs:", result)

    for choice, token_id in result.items():
        decoded = tokenizer.decode(
            [token_id],
            skip_special_tokens=False,
            clean_up_tokenization_spaces=False,
        )
        print(
            f"  {choice}: "
            f"token_id={token_id}, "
            f"decoded={decoded!r}"
        )

    return result


@torch.no_grad()
def predict_one_with_probabilities(row, choice_token_ids):
    model.eval()

    with Image.open(row["path"]) as image_file:
        image = image_file.convert("RGB")

    prompt = render_messages(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=prepare_inference_images(image),
        return_tensors="pt",
    )
    inputs = move_batch(inputs)

    # Qwen/Gemma 최신 구현은 마지막 logits만 반환할 수 있습니다.
    # 지원되지 않는 모델이면 generate 1-step 방식으로 자동 전환합니다.
    try:
        outputs = model(
            **inputs,
            logits_to_keep=1,
            use_cache=False,
        )
        next_token_logits = outputs.logits[0, -1].float()

    except TypeError:
        generated = model.generate(
            **inputs,
            max_new_tokens=1,
            do_sample=False,
            use_cache=True,
            return_dict_in_generate=True,
            output_scores=True,
        )
        next_token_logits = generated.scores[0][0].float()

    token_ids = [
        choice_token_ids[choice]
        for choice in CHOICES
    ]

    # 전체 vocabulary가 아니라 a/b/c/d 네 후보 사이에서 정규화합니다.
    choice_logits = next_token_logits[token_ids]
    probabilities = torch.softmax(choice_logits, dim=0)

    prediction_index = int(probabilities.argmax().item())
    prediction = CHOICES[prediction_index]

    scores = {
        f"p_{choice}": float(probabilities[index].item())
        for index, choice in enumerate(CHOICES)
    }

    sorted_probabilities = torch.sort(
        probabilities,
        descending=True,
    ).values

    scores["margin"] = float(
        (sorted_probabilities[0] - sorted_probabilities[1]).item()
    )

    return prediction, scores


# ------------------------------------------------------------
# test.csv 준비
# ------------------------------------------------------------

test_csv_path = Path(BASE_ROOT) / "test.csv"

test_frame = pd.read_csv(
    test_csv_path,
    dtype={"id": str},
)

# clean_common_frame()에서 answer 컬럼을 요구하므로 임시값 사용
test_frame["answer"] = "a"
test_frame = clean_common_frame(
    test_frame,
    "base_test",
    BASE_ROOT,
)

test_frame["submission_id"] = (
    test_frame["id"]
    .str.replace("base_test::", "", regex=False)
    .astype(str)
)

choice_token_ids = build_choice_token_ids(
    test_frame.iloc[0].to_dict()
)


# ------------------------------------------------------------
# 출력 경로
# ------------------------------------------------------------

ensemble_output_dir = Path(OUTPUT_ROOT)
ensemble_output_dir.mkdir(parents=True, exist_ok=True)

probability_path = (
    ensemble_output_dir
    / f"test_probabilities_{ENSEMBLE_TAG}.csv"
)

partial_path = (
    ensemble_output_dir
    / f"test_probabilities_{ENSEMBLE_TAG}.partial.csv"
)

submission_path = (
    ensemble_output_dir
    / f"submission_{ENSEMBLE_TAG}_softmax.csv"
)


# ------------------------------------------------------------
# 중단된 추론이 있으면 이어서 실행
# ------------------------------------------------------------

records = []

if partial_path.exists():
    previous = pd.read_csv(
        partial_path,
        dtype={"id": str},
    )

    required_columns = {
        "id", "answer",
        "p_a", "p_b", "p_c", "p_d",
    }

    if required_columns.issubset(previous.columns):
        records = previous.to_dict("records")
        print(f"부분 결과에서 재개: {len(records):,}개 완료")
    else:
        print("기존 partial 파일 형식이 달라 새로 시작합니다.")

completed_ids = {
    str(record["id"])
    for record in records
}

started_at = time.time()

for _, row in tqdm(
    test_frame.iterrows(),
    total=len(test_frame),
    desc="Ensemble probability inference",
):
    submission_id = str(row["submission_id"])

    if submission_id in completed_ids:
        continue

    prediction, scores = predict_one_with_probabilities(
        row.to_dict(),
        choice_token_ids,
    )

    records.append({
        "id": submission_id,
        "answer": prediction,
        **scores,
    })

    completed_ids.add(submission_id)

    if len(records) % SAVE_EVERY == 0:
        pd.DataFrame(records).to_csv(
            partial_path,
            index=False,
        )


# ------------------------------------------------------------
# 원본 test 순서로 정렬하고 검증
# ------------------------------------------------------------

probability_frame = (
    pd.DataFrame(records)
    .drop_duplicates("id", keep="last")
)

ordered_ids = (
    test_frame[["submission_id"]]
    .rename(columns={"submission_id": "id"})
)

ordered_ids["id"] = ordered_ids["id"].astype(str)
probability_frame["id"] = probability_frame["id"].astype(str)

probability_frame = ordered_ids.merge(
    probability_frame,
    on="id",
    how="left",
    validate="one_to_one",
)

probability_columns = [
    "p_a", "p_b", "p_c", "p_d",
]

if probability_frame[probability_columns].isna().any().any():
    missing_ids = probability_frame.loc[
        probability_frame["p_a"].isna(),
        "id",
    ].head(10).tolist()

    raise RuntimeError(
        f"확률이 없는 test sample이 있습니다: {missing_ids}"
    )

probability_sums = (
    probability_frame[probability_columns]
    .sum(axis=1)
    .to_numpy()
)

if not np.allclose(
    probability_sums,
    1.0,
    atol=1e-5,
):
    raise RuntimeError(
        "a/b/c/d 확률의 합이 1이 아닙니다."
    )


# ------------------------------------------------------------
# 확률 파일 및 Kaggle 제출 파일 저장
# ------------------------------------------------------------

probability_frame.to_csv(
    probability_path,
    index=False,
)

probability_frame[
    ["id", "answer"]
].to_csv(
    submission_path,
    index=False,
)

elapsed_minutes = (
    time.time() - started_at
) / 60

print(f"완료 시간: {elapsed_minutes:.1f}분")
print("확률 파일:", probability_path)
print("제출 파일:", submission_path)

display(probability_frame.head())

Choice token IDs: {'a': 236746, 'b': 236763, 'c': 236755, 'd': 236753}
  a: token_id=236746, decoded='a'
  b: token_id=236763, decoded='b'
  c: token_id=236755, decoded='c'
  d: token_id=236753, decoded='d'


Ensemble probability inference:   0%|          | 0/6714 [00:00<?, ?it/s]

완료 시간: 33.4분
확률 파일: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/test_probabilities_03_final_other.csv
제출 파일: /content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/submission_03_final_other_softmax.csv


,id,answer,p_a,p_b,p_c,p_d,margin
0,test_0001.jpg,b,0.000018,0.999790,0.000094,0.000098,0.999692
1,test_0002.jpg,c,0.000022,0.000055,0.999810,0.000113,0.999696
2,test_0003.jpg,a,0.484844,0.004483,0.037522,0.473151,0.011693
3,test_0004.jpg,d,0.002698,0.002322,0.388095,0.606885,0.218790
4,test_0005.jpg,c,0.081475,0.006348,0.906205,0.005972,0.824730


## 실행 체크리스트

- 대상 GPU는 **Colab G4 / RTX PRO 6000 96GB** 또는 80GB급 이상이며, 75GiB 미만이면 중단합니다.
- `SMOKE_TEST=False`가 기본값입니다. 전체 학습 전 파이프라인만 확인하려면 잠시 `True`로 바꾸세요.
- 각 Stage 폴더에는 `adapter_model.safetensors`, `adapter_config.json`, `parameter_manifest.csv`, `stage_config.json`, `history.csv`, 검증 예측이 저장됩니다.
- 학습 중에는 모든 Stage에서 100 optimizer update마다 adapter와 `training_state.pt`를 저장하여 자동 재개합니다.
- 최종 Stage가 끝나면 test 추론을 실행하여 `submission.csv`와 `test_predictions.csv`를 저장합니다. test 추론도 100개마다 부분 저장됩니다.
- 결과 루트: `/content/drive/MyDrive/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/` (Drive 미사용 시 `/content/vqa_checkpoints/gemma4_31b_ocr_curriculum_g4/`)
- OOM이면 `MAX_PIXELS=512*512`, `MAX_LENGTH_CANDIDATES=(768, 1024)`, `LORA_R=8` 순서로 낮추세요.

- Every completed Stage saves its own `kaggle_submission.csv`; Stage 3 is also copied to root-level `submission.csv`.